In [ ]:
import pickle
import torch
import matplotlib.pyplot as plt
from functools import partial
from src import models, method, utils

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load vocabulary & embeddings
with open("data/keywords.pkl", "rb") as f:
    keywords = pickle.load(f)
kw_matrix = torch.load("data/keywords_emb_blip2_mean.pt", map_location=device)

# Load models
pipe = models.load_sdxl_pipeline(device=device)
processor, text_model, vision_model = models.load_blip2_components()

In [ ]:
prompt = "A tree whose leaves are made of delicate, colored glass"
seed = 42

drift_keywords, weights, base_img = method.get_drift_keywords(
    pipe=pipe,
    prompt=prompt,
    seed=seed,
    encoding_processor=processor,
    encoding_model=(text_model, vision_model),
    emb_matrix=kw_matrix,
    keywords_list=keywords,
    topk=3
)

print("Detected Drift Keywords & Weights:")
for kw, w in zip(drift_keywords, weights.tolist()):
    print(f"  - {kw}: {w:.4f}")

In [ ]:
# Apply ODC
prompt_embeds, pooled_embeds = method.get_clean_embeddings_qr(
    pipe, prompt, drift_keywords, weights, alpha=1.0, preserve_start=False
)

generator = torch.Generator(device=device).manual_seed(seed)
improved_img = pipe(
    prompt_embeds=prompt_embeds,
    pooled_prompt_embeds=pooled_embeds,
    generator=generator
).images[0]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(base_img); axes[0].set_title("Standard Generation"); axes[0].axis("off")
axes[1].imshow(improved_img); axes[1].set_title("ODC"); axes[1].axis("off")
plt.tight_layout()
plt.show()